In [ ]:
pip install numpy==2.1 pandas numba

In [2]:
import pandas as pd
import numpy as np

def consecutive_max_volume_count(s: pd.Series) -> pd.Series:
    """
    计算每个成交量是前面连续多少个成交量的最大值
    :param s: 成交量时间序列，要求索引为DatetimeIndex
    :return: 连续计数序列
    """
    v = s.values
    n = len(v)
    last_greater = np.full(n, -1)  # 记录最近更大值的位置
    stack = []  # 维护单调递减栈
    counts = np.zeros(n, dtype=int)

    for i in range(n):
        # 弹出栈顶比当前小的元素
        while stack and v[stack[-1]] < v[i]:
            stack.pop()
        
        # 记录最近更大值位置
        if stack:
            last_greater[i] = stack[-1]
        
        # 当前索引入栈
        stack.append(i)
        
        # 计算连续计数
        if i == 0:
            counts[i] = 0  # 首元素无前驱
        else:
            counts[i] = (i - (last_greater[i] + 1)) if last_greater[i] != -1 else i
    
    return pd.Series(counts, index=s.index)

# 使用示例
# 构造测试数据(假设1分钟K线)
date_rng = pd.date_range('2023-01-01 09:00', periods=8, freq='T')
df = pd.DataFrame({
    'volume': [6, 1, 2, 4, 3, 5, 1, 5]
}, index=date_rng)

# 执行计算
df['consecutive_counts'] = consecutive_max_volume_count(df['volume'])

print(df)

                     volume  consecutive_counts
2023-01-01 09:00:00       6                   0
2023-01-01 09:01:00       1                   0
2023-01-01 09:02:00       2                   1
2023-01-01 09:03:00       4                   2
2023-01-01 09:04:00       3                   0
2023-01-01 09:05:00       5                   4
2023-01-01 09:06:00       1                   0
2023-01-01 09:07:00       5                   1


/var/folders/w7/j3n3qwq96dbgf9dfwmzg80n80000gn/T/ipykernel_26307/2503477063.py:38: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_rng = pd.date_range('2023-01-01 09:00', periods=8, freq='T')
